# Uplift Modeling in Marketing — Notebook 3: Meta-learners (S/T/X/R)

> Continuation of `02_Baseline_Propensity_PT.ipynb`. This notebook does not share the
> kernel of the previous one — reload below the training/validation splits and the
> response-targeting baseline of S3.2 (whose score enters the comparative table of S4.3).
> The sealed test remains hidden: this notebook never touches it.

---

## Contents

- [Setup — Retaking from S1-S3](#setup)
- [Section 4 — Meta-learners (S/T/X/R)](#s4)
    - [4.1 Adjustment of the Four Meta-Learners](#s4-1)
    - [4.2 Distribution of Estimated CATE by Learner](#s4-2)
    - [4.3 Preliminary Evaluation with Uplift Metrics (Validation)](#s4-3)
    - [4.4 Diagnosis of Heterogeneity: Signal or Sampling Noise?](#s4-4)
        - [4.4.1 Sealed Test Floor](#s4-4-1)
        - [4.4.2 GATES-lite: Does the CATE ranking separate groups with different ATE?](#s4-4-2)
    - [4.5 Iteration of Features and Regularization (X/R-Learner)](#s4-5)
    - [4.6 Sensitivity to Base Algorithm](#s4-6)
    - [4.7 Ablation: Known vs. Internally Estimated Response Targeting (X/R)](#s4-7)
    - [4.8 Isolating Bagging: Single Tree vs. Random Forest, Same Depth](#s4-8)
    - [4.9 Robustness under repeated resampling](#s4-9)
        - [4.9.1 Stability of `max_depth` (3/4/5)](#s4-9-1)
        - [4.9.2 Pared Pair Comparison: Leader vs Strong Competitors Seen Before](#s4-9-2)
        - [4.9.3 The Two Leaders Against the Response-Targeting Baseline](#s4-9-3)
        - [4.9.4 Why Does the Qini AUC Vary So Much Between Samples?](#s4-9-4)
    - [4.10 Linear S-Learner with Explicit T×X Interactions (Optional, Interpretable Baseline)](#s4-10)
    - [Synthesis of Section 4](#s4-summary)
- [Glossary (Quick Reference)](#glossary)

---

In [ ]:
import sys
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.exceptions import ConvergenceWarning

# Path bootstrap: allows `from src...` from the notebooks directory.
PROJECT_ROOT = Path('..').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import SEED
from src.i18n import make_lang
from src.viz import apply_plot_style

np.random.seed(SEED)
pd.set_option('display.max_columns', 50)
pd.set_option('display.precision', 4)
warnings.filterwarnings('ignore', category=FutureWarning)
# The internal causalml meta-learner propensity model (S4) uses an
# elastic-net solver that does not converge within the default max_iter on
# small samples. This does not affect the result because it is only a
# nuisance model, but it pollutes the output.
warnings.filterwarnings('ignore', category=ConvergenceWarning)

# Idioma canônico deste notebook é PT — passthrough, sem chamada de rede.
# EN edition: only this line is switched to make_lang('en').
lang = make_lang('en')

apply_plot_style()

In [ ]:
# Stack causal (verifique a instalação antes da primeira run)
# pip install econml causalml scikit-uplift shap mlflow

# Meta-learners e árvores de uplift
from causalml.inference.meta import BaseSRegressor, BaseTRegressor, BaseXRegressor, BaseRRegressor
from causalml.inference.tree import UpliftRandomForestClassifier

# Causal Forest com intervalos de confiança
from econml.dml import CausalForestDML
from econml.metalearners import TLearner, SLearner, XLearner

# Avaliação
from sklift.metrics import uplift_at_k, qini_auc_score, uplift_auc_score

from src.compat import patch_sklearn_matplotlib_support
patch_sklearn_matplotlib_support()
from sklift.viz import plot_qini_curve, plot_uplift_curve

# Interpretabilidade
import shap

In [ ]:
from src.config import ARTIFACTS_DIR, RETRAIN

ARTIFACTS_DIR.mkdir(exist_ok=True, parents=True)

# MLflow — ajuste o URI conforme seu setup Windows
# mlflow.set_tracking_uri('file:///C:/path/to/mlruns')
# mlflow.set_experiment('uplift_hillstrom')

<a id="setup"></a>

## Setup — Retaking from S1-S3

In the normal flow, `get_train_val` loads the persisted training/validation manifests by `02_Baseline_Propensity_PT` (`train_index.parquet`, `validation_index.parquet`, `dataset_manifest.json`) and validates the SHA-256 fingerprint and `n_rows` of the current dataset against the manifest — without rewriting the sealed test index, which is only written by notebook 2. Recalculating via `train_test_split` is only the bootstrap fallback, used when no split has been persisted yet; once a sealed test exists, this fallback never happens silently (see `src/splits.py`). We also re-adjust the response-targeting baseline of S3.2, cheap enough not to be worth serializing.

In [ ]:
from src.config import POOLED_TREATMENT_COL, PRIMARY_OUTCOME
from src.data import add_pooled_treatment, load_hillstrom
from src.learners import fit_propensity_baseline, predict_propensity_score
from src.splits import get_train_val

df = load_hillstrom()
df_pooled = add_pooled_treatment(df)
train_df, val_df = get_train_val(df_pooled, persist_test=False)

propensity_model = fit_propensity_baseline(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME)
val_df['propensity_score'] = predict_propensity_score(propensity_model, val_df)

labels = lang({'header': 'Tamanho das partições (retomado de S3)'})
print(f"{labels['header']}:")
print(f"  Training:   {len(train_df):>6} rows | treated: {int(train_df[POOLED_TREATMENT_COL].sum()):>6}")
print(f"  Validation: {len(val_df):>6} rows | treated: {int(val_df[POOLED_TREATMENT_COL].sum()):>6}")

<a id='s4'></a>
<a id="s4"></a>

# Section 4 — Meta-learners (S/T/X/R)


The first objective here will be to estimate CATE with the four classic meta-learners — S-, T-, X- and R-learner — using **a single base learner (LightGBM)** for all. Varying meta-learner and base learner at the same time would make the comparison uninterpretable: if one wins, we wouldn't know if it was the strategy of meta-learning or the capability of the base model.

The next step will be the binary outcome (`visit`) is treated via variants **Regressor** (regression of probability) in the four learners — not Classifier. Reason: the R-learner of `causalml` only exists as Regressor; using Classifier in the other three would create an asymmetry of capacity between learners, which would cost more to the comparison than it's worth the gain of using the more "correct" variant for classification in three of them.

And then, produce CATE out-of-sample in validation for the four, compare with the response-targeting baseline (S3) by the same uplift metrics, and visualize the shape of the distribution of each estimator.

**Treatment:** continues pooled (S3–S6), as established in S3.

> ### Registered hypothesis before running
> The **X-learner has structural advantage** in this dataset, because it was designed specifically for treatment arms with imbalanced treatment — and here the pooled is 2:1 (treated vs. control). Honest verification of this hypothesis, with real numbers, comes in Section 4.3. If it doesn't hold, that's what will be reported (Absolute Rule #6: negative result is result).


**Note added after the result of 4.3.** No meta-learner beat the baseline with a margin — that raised the question of whether the treatment effect is close to homogeneous in this population, or if the estimators (LightGBM vanilla, raw features) weren't capturing real heterogeneity. Three sub-sections were added to investigate, in this order: **4.4** (diagnosis: real signal or noise?), **4.5** (derived features and regularization, focused on X/R) and **4.6** (sensitivity to base algorithm). None of them reopens or re-adjusts what was already decided in 4.1-4.3 — all compare against the original results, side by side.

<a id='s4-1'></a>
<a id="s4-1"></a>

## 4.1 Adjustment of the Four Meta-Learners

Categorical variables (`zip_code`, `channel`, `history_segment`) require a different treatment than that used in the S3 baseline: `causalml` converts `X` to a pure numpy array internally before passing it to the base learner, which discards the pandas category dtype that LightGBM relies on to natively handle categorical variables. Therefore, they are encoded via one-hot (adjusted only for training) before entering the four meta-learners.

The `RETRAIN` pattern of the project is followed: as this is the first time these models are adjusted, this cell trains and serializes to `artifacts/meta_learners.joblib` even with `RETRAIN=False` (bootstrap behavior, to avoid blocking the first execution); future re-executions load the saved artifact instead of retraining, unless `RETRAIN=True`.

In [ ]:
from src.config import META_LEARNERS_PATH
from src.learners import get_meta_learners, predict_meta_learners_uplift

meta_models, meta_encoder = get_meta_learners(
    train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, RETRAIN, META_LEARNERS_PATH,
)
meta_cate = predict_meta_learners_uplift(meta_models, val_df, meta_encoder)
for name, cate in meta_cate.items():
    val_df[f'{name.lower()}_learner_uplift'] = cate

labels = lang({'header': 'CATE estimado — resumo por learner (validação)'})
print(f"{labels['header']}:")
print(val_df[[f'{n.lower()}_learner_uplift' for n in meta_cate]].describe().T)

<a id='s4-2'></a>
<a id="s4-2"></a>

## 4.2 Distribution of Estimated CATE by Learner

Before comparing metrics, it's worth seeing the shape of each estimator's distribution — meta-learners have known bias/variance trade-offs (S-learner tends to shrink treatment effects when treatment is just another feature among many; T-learner tends to have more variance by fitting two fully independent models).

In [ ]:
from src.viz import plot_uplift_distributions

labels = lang({
    'title': 'Distribuição do CATE estimado por meta-learner',
    'subtitle': 'S-learner encolhe efeitos (desvio menor); T-learner é o mais disperso',
})
fig, ax = plot_uplift_distributions(meta_cate, title=labels['title'], subtitle=labels['subtitle'])
plt.show()

>**Insights:** The four distributions have a mean very close to (0.058–0.060), consistent with the pooled ATE of ~6pp estimated in S2 — CATE should, on average, recover the ATE. What differentiates the four is the **dispersion**: the S-learner is visibly the most concentrated (standard deviation 0.036, range -0.15 to +0.42), exactly the expected behavior of an estimator that shares a single model between arms and tends to shrink heterogeneity. The T-learner is the most dispersed (standard deviation 0.070, range -0.35 to +0.48) — two fully independent models accumulate more noise in the difference. The X- and R-learners fall between the two extremes (standard deviations 0.050 and 0.066, respectively).

<a id='s4-3'></a>
<a id="s4-3"></a>

## 4.3 Preliminary Evaluation with Uplift Metrics (Validation)

Same ruler as S3.3 — Qini AUC, uplift AUC (AUUC), and uplift@30% — now including the four meta-learners alongside the response-targeting baseline. This is not yet the formal comparison of S6 (without bootstrap of IC, without opening the sealed test): it's a first reading to guide what to investigate in S5–S6. The fast T-learner of S3.4 is not included here, as already stated — it was just a provisional check.

In [ ]:
from src.evaluation import evaluate_multiple_rankings

all_scores = {
    'Baseline (propensão)': val_df['propensity_score'],
    'S-learner': meta_cate['S'],
    'T-learner': meta_cate['T'],
    'X-learner': meta_cate['X'],
    'R-learner': meta_cate['R'],
}
comparison_table = evaluate_multiple_rankings(
    val_df[PRIMARY_OUTCOME].values, all_scores, val_df[POOLED_TREATMENT_COL].values,
)
print(comparison_table.round(4))

>**Insights:** 
>
>**The X-learner hypothesis did not hold.** None of the four meta-learners beat the response-targeting baseline by a clear margin in this validation.
>
>**S-learner.** Qini AUC 0.0415 surpasses the baseline (0.0395) by a minimal margin — in the third decimal place, small enough to not sustain a model choice based solely on this holdout alone; the stability of this result is investigated further in the development data (4.9).
>
>**X-learner.** Qini AUC 0.0243 is **below** the baseline, contradicting the hypothesis recorded in S4 of structural advantage under imbalanced arms. Reported as is: the hypothesis did not hold.
>
>**T-learner.** Qini AUC 0.0067 is the worst of the four, by a large enough margin (almost 6x smaller than the baseline) to not be just noise in the third decimal place — even without a formal confidence interval yet.
>
>**R-learner.** Qini AUC 0.0196 is in the middle, also below the baseline.
>
>A plausible explanation — not proven here, but consistent with the literature on meta-learners (Künzel et al., 2019) — is that X- and R-learners inherit noise from the first-stage models: the X-learner constructs its pseudo-effects from the same outcome regressions that make the T-learner perform poorly alone (see `BaseXRegressor.fit` in `causalml`: `d_c`/`d_t` are calculated from the predictions of `mu_c`/`mu_t`). The S-learner, using a single shared model between the arms, has less variance to inherit — and that, apparently, outweighs the heterogeneity it may be compressing.
>
>No protocol adjustment was made to produce a different winner (Rule Absolute #6). The stability of the candidates will be investigated further in the development data; S6 will be reserved for the confirmatory evaluation of the final configuration, already selected and frozen at the end of S5.

<a id='s4-4'></a>
<a id="s4-4"></a>

## 4.4 Diagnosis of Heterogeneity: Signal or Sampling Noise?

No meta-learner beat the response-targeting baseline with margin (S4.3) — this can mean two very different things: (a) the treatment effect is close to homogeneous in this population (treat those who already have a natural higher propensity and it captures most of the uplift), or (b) the estimated rankings contain structure associated with differences in effect, although the diagnostics below do not constitute a formal test of heterogeneity against a homogeneous non-zero ATE. The two sections below are complementary diagnostics, not independent tests — they use the same data and investigate related aspects of the same structure, within training/validation; the sealed test is not touched here.

> #### Recorded interpretation rule before running — later qualified
> The initially recorded operational rule was: if the real standard deviation of at least one learner exceeds the 95th percentile of the noise floor by permutation (4.4.1), interpret this as evidence of heterogeneity above sampling noise. After inspecting the null generated by permutation, this reading was qualified: the procedure tests a world where τ(x)=0 for all x, also destroying the ATE — therefore, exceeding this floor indicates structure beyond the totally null null, but does not constitute, by itself, formal evidence of heterogeneity against H0: τ(x)=ATE (homogeneous non-zero effect). If the real standard deviation does not exceed the 95th percentile in any learner, the original reading remains: evidence in favor of homogeneous effect (or not capturable with the current data/features) — declared as a limitation, without forcing a conclusion.

<a id="s4-4-1"></a>

### 4.4.1 Sealed Test Floor

We permute `treatment` in training (intentionally breaking any real association between treatment and outcome — not just heterogeneity, but also the ATE), re-adjust only the S-learner (the cheapest of the four) and measure `std(CATE)` in real validation. We repeat this 20 times — the resulting distribution is what would be observed of CATE dispersion under a completely null association between treatment and outcome (H0: τ(x)=0 for all x), only sampling noise and overfitting of the model. We compare this distribution against the real dispersion of each learner, already measured in 4.2.

**What this test is not:** by permuting the treatment, the null destroys the ATE along with heterogeneity — it is not a formal test of heterogeneity against a homogeneous non-zero effect (H0: τ(x)=ATE), which would require preserving the ATE and perturbing only the dependence on X. We treat this explicitly as an exploratory diagnosis, not a heterogeneity test proper.

Scope: we test only the S-learner by cost — it is an explicit decision, not a silent shortcut. T/X/R have real dispersion even greater than that of the S-learner; if the S-learner already surpasses its own sealed test floor, it is plausible (not proven here) that the other three also surpass theirs.

In [ ]:
from src.evaluation import permutation_noise_floor

permuted_stds = permutation_noise_floor(
    train_df, val_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, meta_encoder,
    learner_name='S', n_reps=20,
)
real_stds = {name: float(cate.std()) for name, cate in meta_cate.items()}

labels = lang({'header': 'Chão de ruído por permutação (S-learner, 20 repetições)'})
print(f"{labels['header']}:")
print(f"  min={permuted_stds.min():.4f}  p50={pd.Series(permuted_stds).median():.4f}  "
    f"p95={pd.Series(permuted_stds).quantile(0.95):.4f}  max={permuted_stds.max():.4f}")
print(f"\nReal deviations by learner: {({k: round(v, 4) for k, v in real_stds.items()})}")

In [ ]:
from src.viz import plot_permutation_noise_floor

labels = lang({
    'title': 'Chão de ruído por permutação vs. dispersão real do CATE',
    'subtitle': 'O desvio real do S-learner (0.036) fica acima do chão de ruído (máx. 0.029)',
})
fig, ax = plot_permutation_noise_floor(permuted_stds, real_stds, title=labels['title'], subtitle=labels['subtitle'])
plt.show()

>**Insights:** 
>
>The permutation noise floor (S-learner, 20 repetitions) ranges from 0.0166 to 0.0290 (p95 = 0.0273, mean 0.0228) — this is the dispersion of CATE that the S-learner itself would produce under a null that completely destroys the treatment-outcome association (not just heterogeneity — the ATE as well), solely due to sampling noise. The real standard deviation of the S-learner (0.0359, measured in 4.2) is **above the maximum observed in the noise floor**, and well above the 95th percentile (0.0273).
>
>The observed CATE dispersion exceeds that produced under a null that completely destroys the treatment–outcome association, suggesting structure beyond pure noise. This diagnosis, however, does not constitute a formal test of heterogeneity against a homogeneous non-zero treatment effect — permuting the treatment zeros the ATE along with any heterogeneity, so surpassing this noise floor is consistent with both "there is real heterogeneity" and "the ATE is non-zero and the model captures some estimation noise different under ATE≠0 vs. ATE=0" without implying τ(x) actually varies by X. We do not distinguish between these two hypotheses here.
>
>The other three learners have even greater real dispersion (T=0.0705, X=0.0500, R=0.0661) — we did not test their individual noise floors (cost), but since all exceed the S-learner's real dispersion, it is plausible that they also exceed their respective noise floors. This was not verified directly here, and we do not treat this extrapolation as a conclusion — it's just an indication.
>
>**This qualifies, but does not resolve, the reading of S4.3.** The weak result of Qini AUC cannot be taken as confirmation of homogeneous effect solely based on this test — there is structure above pure noise being captured, but whether this structure is genuine heterogeneity (τ(x) varying by X) or a reflection of a non-zero ATE with asymmetric estimation noise, this diagnosis does not decide. A test that preserves the ATE and isolates only the dependence on X would remain for future work (not implemented in this round). This distinction motivates trying better features and regularization (S4.5) before discarding the meta-learners — not because heterogeneity is confirmed, but because the current result does not allow us to discard it.

<a id="s4-4-2"></a>

### 4.4.2 GATES-lite: Does the CATE ranking separate groups with different ATE?

Grouped Average Treatment Effects (Chernozhukov et al., 2018), simplified: we rank the validation by the estimated CATE of each learner, cut into quintiles, and estimate the ATE **within each quintile** (via `effects.ate_binary`, the same function as S2.5), with 95% CI. If the quintile with the highest predicted CATE has a real ATE that is higher and distinguishable from the quintile with the lowest predicted CATE (non-overlapping CIs), the ranking carries useful signal of heterogeneity — even if the aggregated Qini metric does not show a clear advantage over the response-targeting baseline.

In [ ]:
from src.evaluation import gates_by_cate_quintile

gates_tables = {
    name: gates_by_cate_quintile(val_df, f'{name.lower()}_learner_uplift', PRIMARY_OUTCOME, POOLED_TREATMENT_COL)
    for name in meta_cate
}
for name, gates_df in gates_tables.items():
    print(f"\n{name}-learner:")
    print(gates_df.round(4))

In [ ]:
from src.viz import plot_gates_bars

labels = lang({
    'title': 'GATES-lite — ATE por quintil de CATE estimado (S-learner)',
    'subtitle': 'ATE cresce do quintil 0 ao 4 (0.049 -> 0.105), mas os ICs quase se tocam',
})
fig, ax = plot_gates_bars(gates_tables['S'], title=labels['title'], subtitle=labels['subtitle'])
plt.show()

>**Insights:** 
>
>In the four learners, the ATE within the quintile of the highest CATE predicted (group 4) is the highest among the five groups, and the trend between the groups is approximately increasing. The pattern is suggestive of the rankings carrying some information about differences in effect, although the learners are evaluated on the same individuals and their rankings are correlated — not four independent confirmations (the complete caveat is in the reading of `gates_delta_bootstrap`, below). For example, in the S-learner, the ATE goes from 0.049 (group 0) to 0.105 (group 4).
>
>However, by the strict criterion of non-overlapping CI, the evidence is **limited, not conclusive**: in the S-learner, the CI of group 0 is [0.021, 0.078] and that of group 4 is [0.077, 0.133] — the intervals almost do not touch (overlap by a fraction of 0.0008), a case of a boundary. In the other three learners (T, X, R), the overlap between group 0 and group 4 is greater, not allowing to declare "distinguishable" with confidence. With ~2.560 lines per quintile and a modest baseline visit rate, the statistical power to separate quintiles is limited — this is a limitation of the data, not a defect of the method.
>
>It is concluded that the two diagnoses point in the same direction, but neither of them is a formal test of heterogeneity: 4.4.1 shows that the real CATE dispersion exceeds the noise floor of a null that destroys all treatment-outcome association (does not distinguish heterogeneity from a homogeneous non-zero ATE — see caveat in the cell above), and 4.4.2 shows a pattern of increasing ATE by CATE quintile in the four learners, but limited by strict IC. Together, they are consistent indications that **something beyond pure noise is present and the ranking by CATE seems to carry some signal**, without confirming genuine heterogeneity (τ(x) varying by X) in a conclusive way. This qualifies the original question: the weak result of Qini AUC of S4.3 cannot be read as confirmation of homogeneous effect, but also we do not have formal evidence to the contrary — which motivates testing better features and regularization (S4.5) and sensitivity to the base algorithm (S4.6) before discarding the meta-learners, treating the question of real heterogeneity vs. homogeneous ATE as an open limitation, not resolved in this section.

In [ ]:
from scipy.stats import spearmanr

from src.evaluation import gates_delta_bootstrap

gates_delta_results = {}
for name in meta_cate:
    col = f'{name.lower()}_learner_uplift'
    result = gates_delta_bootstrap(val_df, col, PRIMARY_OUTCOME, POOLED_TREATMENT_COL, n_groups=5, n_boot=2000)
    rho, rho_p = spearmanr(gates_tables[name]['group'], gates_tables[name]['ate'])
    result['spearman_rho'] = rho
    result['spearman_p'] = rho_p
    gates_delta_results[name] = result

labels = lang({'header': 'Δ_GATES = ATE(quintil topo) − ATE(quintil base), IC 95% via bootstrap (n_boot=2000)'})
print(f"{labels['header']}:\n")
for name, r in gates_delta_results.items():
    print(f"{name}-learner: Δ_GATES={r['delta_gates']:.4f}  IC95%=[{r['ci_low']:.4f}, {r['ci_high']:.4f}]  "
          f"p={r['p_value']:.4f}  |  tendência monótona: rho={r['spearman_rho']:.3f} p={r['spearman_p']:.4f}")

>**Insights:** 
>
>**Δ_GATES Directly Confirms Signal in 2 of 4 Learners, but Don't Treat This as 4 Independent Evidence.** Instead of just checking if the top/base ICs "almost touch" (indirect test, used above), `gates_delta_bootstrap` estimates the distribution of Δ_GATES = ATE(quintile 4) − ATE(quintile 0) directly, via bootstrap within each quintile (appropriate for randomized design — re-sampling preserves the random assignment mechanism within each fixed stratum):
>
>| Learner | Δ_GATES | IC 95% | p-Value | Monotonic Trend (Spearman, Quintile × ATE) |
>|---|---|---|---|---|
>| S | **0.0557** | [0.0153, 0.0950] | **0.0080** | ρ=0.600, p=0.2848 |
>| T | 0.0170 | [-0.0247, 0.0597] | 0.4110 | ρ=0.600, p=0.2848 |
>| X | **0.0434** | [0.0052, 0.0835] | **0.0250** | ρ=0.400, p=0.5046 |
>| R | 0.0263 | [-0.0143, 0.0691] | 0.2170 | ρ=0.900, p=0.0374 |
>
>For S and X, the IC excludes zero — the quintile with the highest predicted CATE has a real ATE distinguishable from the quintile with the lowest predicted CATE, a more direct (and better powered) result than the informal IC overlap check made above. For T and R, the IC includes zero — there is no direct evidence of separation between top and base in these two. The monotonic trend (Spearman between quintile index and quintile ATE) has very low power with only 5 points per learner — none pass the suggestive threshold, and the only one with p<0.05 (R, ρ=0.900) is exactly the learner whose Δ_GATES is not significant, illustrating that "monotonic trend" and "extremes distinguishable" are different readings of the same GATES, not redundant.
>
>**This Is Not Four Independent Pieces of Evidence.** The four learners are evaluated on the same ~12,800 individuals in `val_df`, and their CATE rankings are correlated with each other (all use the same 8 covariates and the same outcome) — a real signal detected by one learner tends to appear, with varying strength, in the others as well, precisely because they are not independent samples. Seeing 2 of 4 significant results should not be read as "50% of the models confirm heterogeneity" in the statistical sense of independent tests — it's more correct to

<a id='s4-5'></a>
<a id="s4-5"></a>

## 4.5 Iteration of Features and Regularization (X/R-Learner)

X- and R-learners fell below the response-targeting baseline in 4.3 (0.0243 and 0.0196 vs. 0.0395). The diagnostics in 4.4 showed structure above the null of pure noise and suggestive signal in the rankings, without formally confirming heterogeneity against a non-zero homogeneous ATE — nonetheless, it's worth trying two known interventions in the literature before discarding these two learners: **derived features** (giving the model structure that it doesn't reconstruct by itself from the raw covariates) and **regularization** (containing the amplification of noise from the first stage, a structural problem of X/R documented in Künzel et al., 2019 and Nie & Wager, 2021). S- and T-learners **do not** enter this iteration — they remain as an unaltered reference of 4.1-4.3.

**Derived Features** (`src/features.py`, isolated module — never touches global `FEATURE_COLS`/`CAT_VARS`; S3 and the four original learners of S4 remain identical if re-executed):(1) `history_per_recency = history / (recency + 1)` — genuine ratio, not reconstructible by splits in the two isolated variables; distinguishes a high-value "dormant" client (high history, high recency) from an active one (high history, low recency); (2) `newbie_x_channel` (6 levels) — acquisition channel can proxy different digital engagement for new vs. established clients; (3) `mens_and_womens` (binary) — clients who bought in both categories may be a distinct profile (buyers for third parties/families); (4) `zip_code_x_channel` (9 levels) — combination can proxy a digital engagement segment that neither variable captures alone.

**Regularization** (`_regularized_base_learner`): `num_leaves=15` (default 31), `max_depth=5` (default no limit), `min_child_samples=50` (default 20), `reg_alpha=reg_lambda=1.0` (default 0) — a final tree that is more shallow and conservative, isolating the axis of complexity without changing `n_estimators`/`learning_rate`.

> #### Registered Structure (No Adoption Threshold)
> Exactly 3 configurations, no grid search: **(A)** vanilla features + hyperparameters; **(B)** original features + regularization; **(C)** both combined. The three enter the same comparative table of 4.3, alongside the baseline and the four original learners — without overwriting anything. There is no numerical threshold of improvement required to "win": the best Qini AUC validation among all lines follows as a candidate for S5/S6, exactly as the S-learner was treated as the best among the 4 in 4.3 — the formal distinction (whether a difference is real or noise) remains for the IC of bootstrap of S6.

**Cost:** Adjusting X and R under the 3 configurations (6 fits in total, over 38,400 lines of training) takes the order of minutes — measured below, in the very execution cell, not estimated. No cache/RETRAIN here: these are diagnostic iterations, not a model that will be loaded in future sections.

In [ ]:
from src.features import EXTENDED_BIN_VARS, EXTENDED_CAT_VARS, EXTENDED_CONT_VARS, add_engineered_features
from src.learners import _default_base_learner, build_meta_learner_encoder, fit_meta_learners_regularized

train_ext = add_engineered_features(train_df)
val_ext = add_engineered_features(val_df)
encoder_ext = build_meta_learner_encoder(train_ext, cat_vars=EXTENDED_CAT_VARS)

s45_timing = {}

# Config A: features estendidas + hiperparâmetros vanilla (isola o efeito das features)
t0 = time.time()
models_a = fit_meta_learners_regularized(
    train_ext, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, encoder_ext,
    cont_vars=EXTENDED_CONT_VARS, bin_vars=EXTENDED_BIN_VARS, cat_vars=EXTENDED_CAT_VARS,
    base_learner_factory=lambda: _default_base_learner(SEED),
)
s45_timing['A'] = time.time() - t0
cate_a = predict_meta_learners_uplift(
    models_a, val_ext, encoder_ext,
    cont_vars=EXTENDED_CONT_VARS, bin_vars=EXTENDED_BIN_VARS, cat_vars=EXTENDED_CAT_VARS,
)

# Config B: features originais + hiperparâmetros regularizados (isola o efeito da regularização)
t0 = time.time()
models_b = fit_meta_learners_regularized(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, meta_encoder)
s45_timing['B'] = time.time() - t0
cate_b = predict_meta_learners_uplift(models_b, val_df, meta_encoder)

# Config C: features estendidas + hiperparâmetros regularizados (combinação)
t0 = time.time()
models_c = fit_meta_learners_regularized(
    train_ext, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, encoder_ext,
    cont_vars=EXTENDED_CONT_VARS, bin_vars=EXTENDED_BIN_VARS, cat_vars=EXTENDED_CAT_VARS,
)
s45_timing['C'] = time.time() - t0
cate_c = predict_meta_learners_uplift(
    models_c, val_ext, encoder_ext,
    cont_vars=EXTENDED_CONT_VARS, bin_vars=EXTENDED_BIN_VARS, cat_vars=EXTENDED_CAT_VARS,
)

s45_scores = {
    'Baseline (propensão)': val_df['propensity_score'],
    'S-learner (original)': meta_cate['S'],
    'T-learner (original)': meta_cate['T'],
    'X-learner (original)': meta_cate['X'],
    'R-learner (original)': meta_cate['R'],
    'X-learner (A: features)': cate_a['X'],
    'R-learner (A: features)': cate_a['R'],
    'X-learner (B: regularizado)': cate_b['X'],
    'R-learner (B: regularizado)': cate_b['R'],
    'X-learner (C: features+reg)': cate_c['X'],
    'R-learner (C: features+reg)': cate_c['R'],
}
s45_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s45_scores, val_df[POOLED_TREATMENT_COL].values)
print(s45_table.round(4))
labels = lang({'header': 'Tempo de fit por configuração (X+R cada)'})
print()
print(f"{labels['header']}: {({k: round(v, 1) for k, v in s45_timing.items()})}")

>**Insights:** 
>
>**No configuration surpasses the original S-learner, but the baseline is revealing.** None of the six combinations (X/R × A/B/C) beat the best Qini AUC seen (S-learner, 0.0415) nor the baseline (0.0395) — the 4.3 candidate remains the same. But the numbers show large and specific effects, not noise.
>
>**Isolated regularization (B) was the best intervention for the X-learner:** Qini AUC rises from 0.0243 (original) to 0.0347 — almost closes the distance to the baseline. Consistent with the hypothesis that the X-learner suffered from amplifying noise from the first stage; a more conservative final tree attacks exactly that.
>
>**Isolated derived features (A) were the best intervention for the R-learner:** Qini AUC rises from 0.0196 (original) to 0.0352 — gain comparable to the regularization in the X-learner, by a different mechanism.
>
>**Each intervention helped the "wrong" learner little or nothing.** Features (A) drastically worsened the X-learner (0.0243 → 0.0013) — plausibly because the two categorical interaction categories (6 and 9 levels) increase the dimensionality of the one-hot, giving more space for the X-learner to overfit its pseudo-first-stage effects, exactly the problem that regularization (not present in this configuration) would mitigate. Regularization (B) helped the R-learner only marginally (0.0196 → 0.0242).
>
>**Combining both (C) was not the sum of the parts — worsened both** compared to their best isolated configuration, and in the R-learner even fell below the original (0.0162 vs. 0.0196). This has no proven explanation here; it is reported as is, without forcing a narrative.
>
>No protocol adjustment was made to produce a winner (Rule Absolute #6) — the six configurations are all in the table above, not just the favorable ones. The useful finding is not "which one won," but that **each meta-learner has a different and specific remedy** (X responds to regularization, R responds to features), and that combining them naively is not safe to assume will work.

<a id='s4-6'></a>
<a id="s4-6"></a>

## 4.6 Sensitivity to Base Algorithm

Up to this point, LightGBM has been the only base algorithm tested — varying meta-learner (S/T/X/R), features (4.5), and regularization (4.5), but always within the *gradient boosting in tree* family. This section tests whether the poor result of S4 is specific to LightGBM or a ceiling that remains when switching to a different family of algorithms — a completely different axis from the previous ones.

Three new families, each with **a single representative** (without duplicating boosting — LightGBM already covers this family):

| Family | Learner | Rationale |
|---|---|---|
| Linear/regularized | `ElasticNet` | Tests whether the CATE is well approximated by an additive/linear surface |
| Single Tree | `DecisionTreeRegressor(max_depth=4)` | Deliberately limited depth — a tree without restriction would explode in variance |
| Ensemble of Trees (bagging) | `RandomForestRegressor` | Mechanically distinct family from boosting (variance reduced by decorrelation, not by sequential gradient) |

Single axis, not crossed with 4.5: original features (the 8 raw covariates), vanilla hyperparameters of the learner itself (only `random_state` fixed), the four meta-learners (S/T/X/R) for each family — the same recipe of 4.1, varying only the base learner.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor
from src.learners import fit_meta_learners

families = {
    'Linear (ElasticNet)': lambda: ElasticNet(random_state=SEED),
    'Árvore única (max_depth=4)': lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED),
    'Random Forest': lambda: RandomForestRegressor(random_state=SEED),
}

family_cate = {}
family_timing = {}
for family_name, factory in families.items():
    t0 = time.time()
    models = fit_meta_learners(
        train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, meta_encoder, base_learner_factory=factory,
    )
    family_timing[family_name] = time.time() - t0
    family_cate[family_name] = predict_meta_learners_uplift(models, val_df, meta_encoder)

s46_scores = {'Baseline (propensão)': val_df['propensity_score']}
for name in ['S', 'T', 'X', 'R']:
    s46_scores[f'{name}-learner (LightGBM)'] = meta_cate[name]
for family_name, cate in family_cate.items():
    for name in ['S', 'T', 'X', 'R']:
        s46_scores[f'{name}-learner ({family_name})'] = cate[name]

s46_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s46_scores, val_df[POOLED_TREATMENT_COL].values)
print(s46_table.round(4))
labels = lang({'header': 'Tempo de fit por família (4 meta-learners cada)'})
print(f"\n{labels['header']}: {({k: round(v, 1) for k, v in family_timing.items()})}")

>**Insights:** 
>
>**The baseline model had a significant impact on the result of this validation.** The three results are well different from each other, and each has an identifiable explanation.
>
>**S-learner with ElasticNet gives Qini AUC = 0.0000 exactly — it's not noise, it's mathematics.** In the source code of `causalml` (`BaseSRegressor.fit`/`predict`), the treatment enters as another column in `X` (`np.hstack((w, X))`), without interaction term with the co-variables. A **linear** model without interactions produces `CATE = mu(x,1) - mu(x,0) = β_tratamento` — a **constant**, equal for everyone. Ranking by a constant score does not discriminate anyone: Qini AUC is zero by construction, not due to lack of signal in the data. This does not happen with T/X/R because they do not share parameters between the arms in the same way.
>
>**A single shallow tree (`max_depth=4`) was the best configuration seen throughout Section 4.** X-learner with a single tree achieves Qini AUC = 0.0627, +58.7% over the response-targeting baseline (0.0395), or 1.59×, and above any result of 4.1-4.5 (the previous best was the S-learner LightGBM, 0.0415). T and S with a single tree also clearly surpass their LightGBM versions. This is consistent with what 4.4 and 4.5 already pointed out — structure beyond pure noise, although not formally confirmed as heterogeneity (see caveat in 4.4.1) — and suggests that **simpler and more conservative models** capture this structure better than vanilla LightGBM — the single tree takes this logic to the extreme (the opposite of "more model capacity is better").
>
>**Random Forest performs poorly, with Qini AUC negative for S and T** (-0.0106 and -0.0096 — worse than random ranking). We do not have a proven explanation here — it is reported as is. A plausible, untested hypothesis: bagging reduces variance between trees, but each individual tree of the Random Forest (by default, without limit of depth) still overfits the training base; the average of several overfit trees does not necessarily correct the shared bias of overfitting, unlike what the intuition of "ensemble reduces variance" would suggest for this specific case of effect estimation.
>
>**This changes the informal leader of Section 4, based on this single-split validation.** Without any protocol adjustment (Rule Absolute #6), the result of the highest Qini AUC validation throughout Section 4 now becomes **X-learner with a single shallow tree (0.0627)**, not the S-learner with LightGBM (0.0415) anymore. This is reported as is — a validation candidate, not a confirmation; without confidence interval, without sealed test. Section 4.9 revisits explicitly if this leadership is stable under resampling — the number 0.0627 itself should not be read as a robust winner until then.

<a id='s4-7'></a>
<a id="s4-7"></a>

## 4.7 Ablation: Known vs. Internally Estimated Response Targeting (X/R)

The Hillstrom is an RCT: by design, $P(T{=}1 \mid X) = P(T{=}1) = 2/3$ (pooled), **does not depend on $X$**. Nevertheless, `fit_single_meta_learner` calls `causalml` without providing `p`, leaving X- and R-learners to estimate the response targeting internally via `ElasticNetPropensityModel` — unnecessary in an RCT, and potentially a source of artificial noise in the nuisance model.

**API Audit (`causalml` 0.15.5, verified in source code, not assumed)** before running anything — the two learners treat `p` in mechanically different ways:

**X-learner.** `p` in `fit` does not affect the models `mu_c/mu_t/tau_c/tau_t` (are adjusted equally, with or without `p`) — the only effect is to prevent `self.propensity_model` from being set. `p` in `predict` **actually matters**: the final combination is `te = p·tau_c_hat + (1-p)·tau_t_hat`. If the `fit` received `p` explicitly and the `predict` is called without `p`, it raises `TypeError` (`self.propensity_model` is `None` by default, not an absent attribute — verified empirically, not `AttributeError` as one might expect). Therefore, the ablation needs to pass `p` in **both fit and predict**, with arrays of the correct size in each call.

**R-learner.** `p` in `fit` enters directly into the R loss (`(y - ŷ)/(w - p)`, weight `(w-p)²`) — actually changes what `models_tau` learns. `p` in `predict` is accepted in the signature but **never referenced in the method body** — dead parameter, without risk of error.

**Hypothesis:** fixing `p = 2/3` (theoretical value of the experimental design) instead of estimating it should reduce nuisance model noise and help X and/or R.

**Single variable changed:** how the response targeting enters fit/predict (internal vs. fixed). Base learner, features, seed — everything else identical between the two configurations. Tested with two base learners (the current leader, shallow tree, and regularized LightGBM 4.5), to check if the result generalizes.

In [ ]:
import numpy as np
from sklearn.tree import DecisionTreeRegressor

from src.learners import encode_meta_learner_features, fit_single_meta_learner, predict_single_meta_learner

tree_factory = lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED)
p_train_fixed = np.full(len(train_df), 2 / 3)
p_val_fixed = np.full(len(val_df), 2 / 3)

X_train_full = encode_meta_learner_features(train_df, meta_encoder)
X_val_full = encode_meta_learner_features(val_df, meta_encoder)
treatment_arr = train_df[POOLED_TREATMENT_COL].to_numpy()
y_arr = train_df[PRIMARY_OUTCOME].to_numpy(dtype=float)

s47_scores = {'Baseline (propensão)': val_df['propensity_score']}
internal_propensity_stats = {}

for name in ['X', 'R']:
    model_a = fit_single_meta_learner(name, X_train_full, treatment_arr, y_arr, seed=SEED, base_learner_factory=tree_factory)
    s47_scores[f'{name}-learner árvore (A: interna)'] = predict_single_meta_learner(name, model_a, X_val_full)

    if hasattr(model_a, 'propensity') and model_a.propensity:
        group = list(model_a.propensity.keys())[0]
        p_internal = model_a.propensity[group]
        internal_propensity_stats[name] = {
            'mean': float(np.mean(p_internal)), 'std': float(np.std(p_internal)),
            'min': float(np.min(p_internal)), 'max': float(np.max(p_internal)),
        }

    model_b = fit_single_meta_learner(
        name, X_train_full, treatment_arr, y_arr, seed=SEED, base_learner_factory=tree_factory, p=p_train_fixed,
    )
    s47_scores[f'{name}-learner árvore (B: fixa 2/3)'] = predict_single_meta_learner(name, model_b, X_val_full, p=p_val_fixed)

labels = lang({'header': 'Dispersão da propensão interna estimada (deveria ser ~constante 2/3 num RCT)'})
print(f"{labels['header']}:")
for name, stats in internal_propensity_stats.items():
    print(f"  {name}-learner: {({k: round(v, 4) for k, v in stats.items()})}")

s47_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s47_scores, val_df[POOLED_TREATMENT_COL].values)
print()
print(s47_table.round(4))

In [ ]:
# Repeats the ablation with the regularized LightGBM from 4.5 to check whether
# the tree result generalizes to another base learner.
from src.learners import _regularized_base_learner

lgbm_reg_factory = lambda: _regularized_base_learner(SEED)

s47_lgbm_scores = {'Baseline (propensão)': val_df['propensity_score']}
for name in ['X', 'R']:
    model_a = fit_single_meta_learner(name, X_train_full, treatment_arr, y_arr, seed=SEED, base_learner_factory=lgbm_reg_factory)
    s47_lgbm_scores[f'{name}-learner LightGBM-reg (A: interna)'] = predict_single_meta_learner(name, model_a, X_val_full)

    model_b = fit_single_meta_learner(
        name, X_train_full, treatment_arr, y_arr, seed=SEED, base_learner_factory=lgbm_reg_factory, p=p_train_fixed,
    )
    s47_lgbm_scores[f'{name}-learner LightGBM-reg (B: fixa 2/3)'] = predict_single_meta_learner(name, model_b, X_val_full, p=p_val_fixed)

s47_lgbm_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s47_lgbm_scores, val_df[POOLED_TREATMENT_COL].values)
print(s47_lgbm_table.round(4))

>**Insights:** 
>
>**Majoritariamente Negative Result, and That's Reported as Is.** The estimated internal propensity has a mean of 0.6671 (exactly matches the treatment rate performed during training) and a standard deviation of 0.0109, varying between 0.444 and 0.900 among individuals — in other words, the `ElasticNetPropensityModel` does not recover a perfect constant (as would be ideal in an RCT), but the variation is modest, not extreme.
>
>With a shallow tree (current leader): X-learner is practically unchanged (0.0627 → 0.0615, -0.0012 difference); R-learner has a small improvement (0.0453 → 0.0459, +0.0006). With regularized LightGBM: X-learner remains **identical** (0.0347 → 0.0347); R-learner improves a bit more (0.0242 → 0.0291, +0.0049).
>
>**Interpretation:** the pattern is consistent across the two base learners — known propensity does not change the X-learner perceptibly (mechanically expected: in the X-learner, `p` will only respond to a combination of two already calculated predictions in the predict; with small internal dispersion, the weight changes little) and helps the R-learner in a small but consistent way (mechanically also expected: in the R-learner, `p` enters directly into the loss function). In none of the four tested cases is the change large enough to alter the leadership of the section — the shallow tree (0.0627) remains ahead of everything.
>
>**This Does Not Invalidate the Nuisance Model Noise Hypothesis from the Literature (Künzel et al., 2019; Nie & Wager, 2021)** — it only isolates that, *specifically* for the nuisance model of propensity, the measured noise here is too small to explain the poor performance of X/R. The hypothesis remains plausible for the nuisance models of **outcome** (`mu_c`/`mu_t`, used by the X-learner to compute `d_c`/`d_t`) — these were not touched by this experiment and continue to be a candidate to explain the observed gap.
>
>**Limitations:** validation comparison, a single seed, no formal test of significance (this is S6's work, with the sealed test). We did not adjust any hyperparameter to compensate for the null result (Absolute Rule #6).

<a id='s4-8'></a>
<a id="s4-8"></a>

## 4.8 Isolating Bagging: Single Tree vs. Random Forest, Same Depth

In 4.6, the Random Forest (vanilla hyperparameters, no depth limit) performed poorly — negative Qini for S/T. However, this comparison **did not isolate** the effect of bagging: the single tree had `max_depth=4`, the RF had `max_depth=None` (unrestricted). The poor performance of the RF could be overfitting due to depth, not bagging itself.

**Hypothesis:** by equalizing the depth (`max_depth=4` in both), the RF should at least match the single tree — bagging reduces variance, and should not worsen an already shallow model.

**Single Variable Changed:** bagging (multiple trees in bootstrap) vs. a single tree, with `max_depth=4` and `random_state` identical in both. All other RF hyperparameters remain at their default `scikit-learn` values — explicitly documented below, not assumed.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor

rf_defaults = RandomForestRegressor().get_params()
labels = lang({'header': 'Hiperparâmetros do RandomForestRegressor mantidos no default'})
print(f"{labels['header']}:")
for k in ['n_estimators', 'max_depth', 'max_features', 'bootstrap', 'min_samples_leaf', 'min_samples_split']:
    print(f"  {k} = {rf_defaults[k]}")

bagging_families = {
    'Árvore única (depth=4)': lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED),
    'Random Forest (max_depth=4)': lambda: RandomForestRegressor(max_depth=4, random_state=SEED),
}

bagging_cate = {}
for family_name, factory in bagging_families.items():
    models = fit_meta_learners(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, meta_encoder, base_learner_factory=factory)
    bagging_cate[family_name] = predict_meta_learners_uplift(models, val_df, meta_encoder)

s48_scores = {'Baseline (propensão)': val_df['propensity_score']}
for family_name, cate in bagging_cate.items():
    for name in ['S', 'T', 'X', 'R']:
        s48_scores[f'{name}-learner ({family_name})'] = cate[name]

s48_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s48_scores, val_df[POOLED_TREATMENT_COL].values)
print()
print(s48_table.round(4))

>**Insights:**
>
>**Bagging does not help uniformly, and in the leader learner (X) it worsens.** With equal depth, the table changes completely compared to 4.6:
>
>| Learner | Single Tree (depth=4) | Random Forest (depth=4) | Difference |
>|---|---|---|---|
>| S | 0.0515 | 0.0521 | +0.0006 (practical tie) |
>| T | 0.0615 | 0.0524 | **-0.0091** |
>| X | **0.0627** | 0.0538 | **-0.0089** |
>| R | 0.0453 | 0.0492 | +0.0039 |
>
>First finding, confirming a suspicion: **the disaster of RF in 4.6 (negative Qini for S/T) was indeed unrestricted depth, not bagging** — with `max_depth=4`, all four learners with RF become positive and reasonable, nothing catastrophic.
>
>Second finding, more interesting: **bagging did not help in this comparison.** For the X-learner (our leader) and the T-learner, the single tree beats the RF by a margin that is not negligible — the hypothesis that "reducing variance should help" did not hold up here. A plausible, unproven explanation: the average over bootstrap re-samples may smooth out the specific threshold splits that a single tree, seeing the entire training set at once, locates with greater precision — this may help mean squared error of prediction without helping (or even worsening) a ranking metric like Qini AUC, which is sensitive to fine ordering of individuals. Important: this specific comparison (single tree vs. RF, both `max_depth=4`) occurred only in this fixed holdout — unlike 4.9.1/4.9.2/4.9.3, it did not pass through the same analysis of robustness under repeated re-sampling, so it should be read as a result of this split, not as a robust property of the dataset.
>
>**Important limitation:** `max_features=1.0` is the default of `RandomForestRegressor` in the installed scikit-learn — i.e., each tree of the RF sees **all** features in each split; the only source of decorrelation between trees is the bootstrap of lines, not feature sub-sampling (different from the classic recipe of Breiman, which also sub-samples features). This experiment isolates "single tree vs. trees in bootstrap, same depth, no feature sub-sampling" — not necessarily the complete recipe of Random Forest. Testing `max_features='sqrt'` explicitly would remain for a future experiment, not done here to not open a new tuning front.
>
>No protocol adjustment was made to produce this result (Rule Absolute #6) — it is reported as is, including the worsening in the X-learner.

<a id='s4-9'></a>
<a id="s4-9"></a>

## 4.9 Robustness under repeated resampling

All comparisons from 4.1 to 4.8 use a **single fixed split** for training→validation — each Qini AUC reported so far is a single number, from a single validation sample. This leaves an open question: is the current leader (X-learner + shallow tree, Qini AUC 0.0627) a result that would hold under other samples, or a specific favorable split?

This section tests this via **repeated stratified holdout**: within `train_df` (never touching `val_df` or the sealed test), we randomly repeat 15 stratified 75%/25% partitions by (treatment × outcome), with seeds `1000+rep`, adjust each candidate on the 75% and evaluate on the 25% — 15 repeated estimates of Qini AUC per candidate, obtained under different resamplings, instead of a single estimate. **It is not k-fold/OOF**: between repetitions, the same row can appear 0, 1, or more times in the evaluation sets — hence the name. We reuse the same 15 splits/seeds across all candidates of the same comparison, allowing paired differences by split (Δ_r = metric(candidate,r) − metric(reference,r)), more informative than comparing isolated aggregated means.

Three comparisons, in the order they were decided: (4.9.1) stability of `max_depth` ∈ {3,4,5}; (4.9.2) paired comparison of the leader against a small set of strong competitors already seen in 4.6-4.8; (4.9.3) the two best candidates of 4.9.2 against the response-targeting baseline, on the same splits. During 4.9, `val_df` remains untouched; all resamplings occur exclusively within `train_df`. Section 4.10 returns to the fixed holdout only for the previously defined exploratory linear baseline, without using this result to select the winner of the section.

<a id="s4-9-1"></a>

### 4.9.1 Stability of `max_depth` (3/4/5)

**Hypothesis:** if `max_depth=4` (choice used in 4.6-4.8) is a genuinely better configuration — not just the one that got lucky with the fixed split — the average Qini AUC under resampling should be consistently higher than `max_depth=3` or `max_depth=5`, with little overlap between the distributions.

**Single variable changed:** tree depth (3 vs. 4 vs. 5) in the X-learner (the current leader) — same data, same resampling protocol, same `random_state` in the tree.

In [ ]:
from sklearn.tree import DecisionTreeRegressor

from src.evaluation import repeated_holdout_summary, repeated_stratified_holdout

depth_candidates = {
    f'depth={d}': ('meta', 'X', (lambda dd=d: DecisionTreeRegressor(max_depth=dd, random_state=SEED)), False)
    for d in [3, 4, 5]
}
depth_results = repeated_stratified_holdout(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, depth_candidates, n_reps=15)
depth_summary = repeated_holdout_summary(depth_results)

labels = lang({'header': 'Estabilidade de max_depth — X-learner, 15 repeated stratified holdouts (só em train_df)'})
print(f"{labels['header']}:")
print(depth_summary.round(4))

>**Insights:** 
>
>**There is no evidence of a stable advantage of any depth over the others in this re-sampling; we maintain `max_depth=4`.**
>
>| Depth | Mean | Median | Standard Deviation | Win Rate |
>|---|---|---|---|---|
>| 3 | 0.0254 | 0.0296 | 0.0124 | 26.7% |
>| 4 | 0.0250 | 0.0259 | 0.0165 | 40.0% |
>| 5 | 0.0269 | 0.0275 | 0.0099 | 33.3% |
>
>The means vary by only 0.0019 between the highest (depth=5, 0.0269) and the lowest (depth=4, 0.0250) — a difference much smaller than the standard deviation of any of them (0.0099 to 0.0165). `depth=5` has the smallest standard deviation (most stable) and the second highest median; `depth=4` has the highest win rate per split (40%) but also the largest standard deviation (least stable) of the three. There is no depth that dominates the others in all criteria simultaneously — exactly the pattern of "difference within re-sampling variability", not real superiority.
>
>**Decision:** following the rule of not choosing solely by the highest mean and preferring the more stable configuration when differences are not clear, we maintain `max_depth=4` — not because the data from this re-sampling recommend it over the other two, but because none of the other two justify themselves better, and `max_depth=4` is already the used and compared configuration in 4.6, 4.7, and 4.8; changing now would be a protocol change not supported by evidence (Absolute Rule #6). Secondary metrics (Uplift AUC, Uplift@30%) show the same practical tie between the three depths.

<a id="s4-9-2"></a>

### 4.9.2 Pared Pair Comparison: Leader vs Strong Competitors Seen Before

**Hypothesis:** If X-learner + shallow tree is genuinely the best architecture seen in S4 (not just the winner of the fixed split in 4.6), it should win the strongest competitors tested in most of the 15 re-sampled splits, with consistently positive paired differences.

**Single Variable Changed:** Which candidate (meta-learner × base learner × response-targeting configuration), with the same 15 splits/seeds reused across the five. Candidates: the leader (X+shallow tree); the T-learner with the same shallow tree — **not** the baseline here: in 4.6/4.8 it was one of the best results in the fixed holdout (Qini AUC 0.0615, almost tied with the leader), so including it tests directly if this holdout unique strength sustains under re-sampling; the S-learner with LightGBM vanilla (best result in 4.1-4.3, before 4.6); the two regularized candidates from 4.5/4.7 (X and R with regularized LightGBM, R with fixed response-targeting 2/3).

In [ ]:
from src.evaluation import paired_deltas
from src.learners import _regularized_base_learner

tree_factory = lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED)
lgbm_reg_factory = lambda: _regularized_base_learner(SEED)

paired_candidates = {
    'X+Tree(depth=4)': ('meta', 'X', tree_factory, False),
    'T+Tree(depth=4)': ('meta', 'T', tree_factory, False),
    'S+LightGBM(vanilla)': ('meta', 'S', None, False),
    'X+LightGBM(regularizado)': ('meta', 'X', lgbm_reg_factory, False),
    'R+LightGBM(regularizado,p=2/3)': ('meta', 'R', lgbm_reg_factory, True),
}
paired_results = repeated_stratified_holdout(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, paired_candidates, n_reps=15)
paired_summary = repeated_holdout_summary(paired_results)

labels = lang({'header': 'Comparação pareada — 15 repeated stratified holdouts (só em train_df)'})
print(f"{labels['header']}:")
print(paired_summary.round(4))

deltas_vs_leader = paired_deltas(paired_results, baseline_candidate='X+Tree(depth=4)')
deltas_vs_leader[['delta_mean', 'delta_median']] *= -1
deltas_vs_leader['prop_delta_positive'] = 1 - deltas_vs_leader['prop_delta_positive']
print('\nΔ = Qini(X+Tree(depth=4)) − Qini(candidato), por split:')
print(deltas_vs_leader.round(4))

>**Insights:** 
>
>**The leader holds up against most competitors, but not against the S-learner with LightGBM vanilla.**
>
>| Candidate | Mean | Median | Win Rate |
>|---|---|---|---|
>| X+Tree(depth=4) | **0.0250** | 0.0259 | **46.7%** |
>| S+LightGBM(vanilla) | 0.0232 | 0.0228 | 26.7% |
>| X+LightGBM(regularized) | 0.0165 | 0.0163 | 6.7% |
>| R+LightGBM(regularized,p=2/3) | 0.0129 | 0.0158 | 20.0% |
>| T+Tree(depth=4) | 0.0039 | 0.0013 | 0.0% |
>
>Paired differences (X+Tree less each competitor, by split):
>
>| vs. | Δ mean | Δ median | P(Δ>0) |
>|---|---|---|---|
>| T+Tree(depth=4) | +0.0211 | +0.0197 | **93%** |
>| R+LightGBM(regularized,p=2/3) | +0.0121 | +0.0069 | 67% |
>| X+LightGBM(regularized) | +0.0085 | +0.0044 | 67% |
>| S+LightGBM(vanilla) | +0.0018 | +0.0013 | **53%** |
>
>Two distinct patterns here, not one. Against the T+Tree — which had been one of the best candidates in the fixed holdout (Qini AUC 0.0615, almost tied with the leader — see 4.6/4.8), but fell to mean 0.0039 in this re-sampling (table above) — and against the two regularized variants of LightGBM (4.5/4.7), the leader wins with moderate to strong consistency (67%-93% of splits, clearly positive delta mean): this advantage shows up more stably in this repeated protocol, not an artifact of the fixed split in 4.6. But against the S-learner with LightGBM vanilla — the best result in the original block 4.1-4.3, before any base algorithm change — the leader's advantage is a nearly honest coin: wins in only 53% of splits, with delta mean (0.0018) small compared to the standard deviation of the distributions (0.0165 and 0.0115). As a secondary diagnosis — not as formal evidence, since the 15 repeated holdouts overlap partially (are not independent samples) — a paired Wilcoxon test and a paired t-test on this pair give p=0.89 and p=0.71, both consistent with "no detectable difference".
>
>**Honest reading:** X-learner + shallow tree and S-learner + LightGBM vanilla form a **top tier without stable separation** in this re-sampling — both clearly ahead of the T-learner and the regularized variants, without the advantage of one over the other sustaining consistently between splits. The Qini 0.0627 observed in the original holdout was not reproduced under repeated holdout and showed strong sensitivity to the sample. Since the repeated protocol uses fewer observations both for training and evaluation, it is not possible to attribute all the difference exclusively to a favorable split. This does not mean that the X+shallow tree architecture is bad — it remains among the two best candidates of all Section 4, just does not have a stable advantage over the other better candidate.

<a id="s4-9-3"></a>

### 4.9.3 The Two Leaders Against the Response-Targeting Baseline

**The Central Question of This Subsection:** Do the uplift models that are leaders consistently show an advantage over the simple response-targeting baseline (what marketing would already do without any incremental effect model) when evaluated on the same resamples?

**Single Variable Changed:** X+Tree(depth=4) and S+LightGBM(vanilla) — the two top-tier candidates from 4.9.2 — against the response-targeting baseline (`fit_propensity_baseline`/`predict_propensity_score`, already used throughout Section 4 as a reference), on the same 15 splits/seeds.

In [ ]:
baseline_candidates = {
    'X+Tree(depth=4)': ('meta', 'X', tree_factory, False),
    'S+LightGBM(vanilla)': ('meta', 'S', None, False),
    'Baseline (propensão)': ('propensity', None, None, False),
}
baseline_results = repeated_stratified_holdout(train_df, POOLED_TREATMENT_COL, PRIMARY_OUTCOME, baseline_candidates, n_reps=15)
baseline_summary = repeated_holdout_summary(baseline_results)

labels = lang({'header': 'Líderes vs. baseline de propensão — 15 repeated stratified holdouts (só em train_df)'})
print(f"{labels['header']}:")
print(baseline_summary.round(4))

deltas_vs_baseline = paired_deltas(baseline_results, baseline_candidate='Baseline (propensão)')
print('\nΔ = Qini(candidato) − Qini(baseline), por split:')
print(deltas_vs_baseline.round(4))

>**Insights:**
>
>**No Consistente Advantage over the baseline, for neither of the two leaders.**
>
>| Candidate | Mean | Median | Win Rate |
>|---|---|---|---|
>| X+Tree(depth=4) | 0.0250 | 0.0259 | 40.0% |
>| S+LightGBM(vanilla) | 0.0232 | 0.0228 | 40.0% |
>| Response-targeting baseline | 0.0209 | 0.0172 | 20.0% |
>
>Paired differences against the baseline, by split:
>
>| Candidate | Δ mean | Δ median | P(Δ>0) |
>|---|---|---|---|
>| X+Tree(depth=4) | +0.0041 | +0.0007 | 53% |
>| S+LightGBM(vanilla) | +0.0024 | +0.0013 | 60% |
>
>In the raw win rate between the three candidates, X+Tree and S+LightGBM win 40% of the repetitions each, while the baseline wins 20%. But in the paired comparison, the advantage is small and inconsistent: X+Tree surpasses the baseline in only 53% of the splits (delta median of just 0.0007 — essentially zero), and S+LightGBM in 60% (delta median 0.0013). As a secondary diagnosis — same caveat as 4.9.2, overlapping samples, not independent —, Wilcoxon and t-test paired give p=0.56/0.42 (X+Tree) and p=0.68/0.54 (S+LightGBM), none near conventional significance.
>
>**Response to the central question of this subsection: no.** In this re-sampling, neither X+Tree(depth=4) nor S+LightGBM(vanilla) show a consistent advantage over the simple response-targeting baseline — the mean advantage exists and is positive, but is small in front of the variability between samples, and the baseline wins almost half the time. This is reported as is, without trying to recover a larger advantage via hyperparameter tuning or new model search (Absolute Rule #6) — the question that motivated the entire Section 4 (do meta-learners beat the response-targeting baseline?) remains unanswered with a robust affirmative with the data and models tested so far.
>
>**Sealed Test Results:**
>
>**No Consistent Advantage over the baseline, for neither of the two leaders.**
>
>| Candidate | Mean | Median | Win Rate |
>|---|---|---|---|
>| UpliftTree(depth=4) | 0.0250 | 0.0259 | 40.0% |
>| S+LightGBM(vanilla) | 0.0232 | 0.0228 | 40.0% |
>| Response-targeting baseline | 0.0209 | 0.0172 | 20.0% |
>
>Paired differences against the baseline, by split:
>
>| Candidate | Δ mean | Δ median | P(Δ>0) |
>|---|---|---|---|
>| UpliftTree(depth=4) | +0.0041

<a id="s4-9-4"></a>

### 4.9.4 Why Does the Qini AUC Vary So Much Between Samples?

A common pattern among the three comparisons above: the standard deviation of the Qini AUC under resampling (typically 0.010-0.017) is large relative to the means themselves (0.02-0.03) and the differences between candidates (often < 0.01). Some probable, non-mutually exclusive reasons are described below.

**Repeated Protocol Sample Size.** Each repetition adjusts with 75% of `train_df` (≈28,800 lines) and evaluates on 25% (≈9,600 lines) — much less than the original fixed split (`train_df` complete, 38,400 lines, evaluated on `val_df`, 12,800 lines). The Qini AUC is sensitive to how the `visit` events (low prevalence) distribute between treated/control in a smaller sample; with fewer observations, this distribution varies more between repetitions.

**`visit` has relatively low/moderate prevalence, combined with a small magnitude incremental treatment effect** (ATE pooled around a few percentage points, measured in S2) — extracting a stable ranking from a small signal, with a metric (Qini AUC) sensitive to the fine ordering of individuals, is inherently more sensitive to sampling noise than estimating aggregated means (like the ATE itself, which has a much smaller standard error).

**The models themselves are re-adjusted every repetition** with a different subset of training data — part of the observed variance is variance of model estimation (especially for the X-learner, which involves two stages of nuisance model), not just variance of the evaluation metric.

None of these hypotheses has been formally isolated here (it would require, for example, fixing the trained model once and varying only the evaluation sample, separating estimation variance from evaluation variance) — recorded as a plausible explanation and open limitation, not as a tested conclusion.

<a id='s4-10'></a>
<a id="s4-10"></a>

## 4.10 Linear S-Learner with Explicit T×X Interactions (Optional, Interpretable Baseline)

In 4.6, S+ElasticNet produced Qini AUC = 0.0000 **exactly** — not because there was no signal, but because the `causalml` S-learner injects treatment as just another column in X, without an interaction term. A linear model without interactions produces `CATE = μ(x,1) − μ(x,0) = β_treatment`, a constant. This subsection tests whether adding explicit `T×X_j` interactions — the simplest way to give a linear model some heterogeneity capacity — recovers part of the signal captured by the shallow tree, while remaining interpretable and low-complexity. Its role is explanatory, not competitive: this is not an attempt to beat the 4.9 leader, but a test of how much simple linear/additive structure exists in the data.

**Methodological issue identified before implementation, through source-code inspection rather than assumption.** `BaseSRegressor.predict` from `causalml` predicts twice — `hstack([zeros, X])` and then `hstack([ones, X])` — changing **only** the treatment column while keeping the rest of `X` identical in both calls. If `T×X_j` columns were precomputed and concatenated to `X` before being passed to `BaseSRegressor`, they would remain fixed in both counterfactual predictions and cancel out exactly in the subtraction. The CATE would remain constant, silently reproducing the same null result from 4.6 without testing any interaction. For that reason, this S-learner is implemented directly in `src/learners.py` (`fit_s_learner_linear_interaction`/`predict_s_learner_linear_interaction_uplift`), outside the `causalml` wrapper: a single `ElasticNet` fitted on the design matrix `[X, T, T×X_1, ..., T×X_p]`, with `CATE(x) = coef_T + Σ(coef_{T×X_j}·X_j)` computed analytically from the coefficients. Because the model is linear, this difference has closed form.

**Caveat on standardization, noted before running.** The features from `encode_meta_learner_features` are not standardized — `recency` (0-12) and especially `history` (0 to ~3,400, mean ~243) have much larger scales than the binary/one-hot features (0/1). For a linear model with L1/L2 penalization (`ElasticNet`), this artificially favors larger-scale features: the same explanatory power requires a smaller absolute coefficient for a large-scale feature, so the L1 penalty tends to zero out smaller-scale features first, not necessarily less relevant features. The same preprocessing from the original S+ElasticNet is preserved (no standardization) to keep the comparison focused on a single changed variable (adding interactions). This means any interaction coefficient that survives regularization must be read with this caveat in mind, not as confirmed feature importance. Preprocessing, interactions, and hyperparameters are not changed at the same time (Absolute Rule #6): `ElasticNet` hyperparameters remain at the same S4.6 defaults (`alpha=1.0, l1_ratio=0.5`), with no grid search.

**Single changed variable:** add `T×X_j` terms to the linear S-learner. Main comparison: S+ElasticNet without T×X (original, 4.6) versus with T×X. Secondary comparison: against S+DecisionTree(depth=4) (4.6/4.8), not against the section leader, only as a reference for how much signal a nonlinear model captures in the same meta-learner format (S).


In [ ]:
from sklearn.linear_model import ElasticNet
from sklearn.tree import DecisionTreeRegressor

from src.config import BIN_VARS, CAT_VARS, CONT_VARS
from src.evaluation import evaluate_multiple_rankings
from src.learners import (
    encode_meta_learner_features, fit_s_learner_linear_interaction, fit_single_meta_learner,
    predict_s_learner_linear_interaction_uplift, predict_single_meta_learner,
)

X_train_s410 = encode_meta_learner_features(train_df, meta_encoder)
X_val_s410 = encode_meta_learner_features(val_df, meta_encoder)
treatment_train_s410 = train_df[POOLED_TREATMENT_COL].to_numpy()
y_train_s410 = train_df[PRIMARY_OUTCOME].to_numpy(dtype=float)

s410_scores = {}

model_no_interaction = fit_single_meta_learner(
    'S', X_train_s410, treatment_train_s410, y_train_s410,
    base_learner_factory=lambda: ElasticNet(random_state=SEED),
)
s410_scores['S+ElasticNet (sem T×X)'] = predict_single_meta_learner('S', model_no_interaction, X_val_s410)

model_interaction = fit_s_learner_linear_interaction(
    X_train_s410, treatment_train_s410, y_train_s410, alpha=1.0, l1_ratio=0.5,
)
cate_interaction = predict_s_learner_linear_interaction_uplift(model_interaction, X_val_s410)
s410_scores['S+ElasticNet (com T×X)'] = cate_interaction

model_tree = fit_single_meta_learner(
    'S', X_train_s410, treatment_train_s410, y_train_s410,
    base_learner_factory=lambda: DecisionTreeRegressor(max_depth=4, random_state=SEED),
)
s410_scores['S+DecisionTree(depth=4)'] = predict_single_meta_learner('S', model_tree, X_val_s410)

s410_table = evaluate_multiple_rankings(val_df[PRIMARY_OUTCOME].values, s410_scores, val_df[POOLED_TREATMENT_COL].values)
print(s410_table.round(4))

n_features = X_train_s410.shape[1]
coef_t = model_interaction.coef_[n_features]
coef_interaction_vals = model_interaction.coef_[n_features + 1:]
cat_feature_names = list(meta_encoder.get_feature_names_out(CAT_VARS))
feature_names = CONT_VARS + BIN_VARS + cat_feature_names
nonzero_idx = coef_interaction_vals.nonzero()[0]

labels = lang({'header': 'Coeficientes do S+ElasticNet com T×X'})
print(f"\n{labels['header']}:")
print(f"  coef_T (efeito base) = {coef_t:.4f}")
print(f"  non-zero interactions from ElasticNet: {len(nonzero_idx)} of {len(coef_interaction_vals)}")
for i in nonzero_idx:
    print(f"    T×{feature_names[i]}: coef={coef_interaction_vals[i]:.6f}")

>**Insights:** 
>
>**A non-trivial signal was recovered, but the mechanism is almost certainly an artifact of scale, not an identified interaction.**
>
>| Configuration | Qini AUC | Uplift AUC | Uplift@30% |
>|---|---|---|---|
>| S+ElasticNet (without T×X) | 0.0000 | 0.0000 | 0.0626 |
>| **S+ElasticNet (with T×X)** | **0.0229** | 0.0134 | 0.0808 |
>| S+DecisionTree(depth=4) | 0.0515 | 0.0313 | 0.0894 |
>
>Adding interactions takes the S-learner linear out of absolute zero: Qini AUC=0.0229 is a real result, in the range that would be "scientifically interesting" (shows that some structure associated with the ranking of effect can be represented by simple linear interactions) — about 45% of what the shallow tree captures in the same format of meta-learner (S), and comparable in order of magnitude to the response-targeting baseline (0.0395) and to several results of 4.5/4.7.
>
>**But the reason behind this number strongly qualifies the reading.** The `ElasticNet` (`alpha=1.0`, without standardization) zeros **17 of the 18** interaction coefficients — including the very own `coef_T` (treatment effect base) falls to 0.0000. Exactly **one** non-zero term remains: `T×history`, with coefficient 0.000118. `history` is, remarkably, the feature of largest raw scale among the 18 (mean ≈243, standard deviation ≈256, maximum ≈3.346 — against 0/1 for all binaries and one-hot, and 0-12 for `recency`) — exactly the pattern that the pre-registered standardization, recorded before running this experiment, predicted: the L1 penalty of `ElasticNet` favors features of large scale, because the same explanatory power requires a smaller absolute coefficient value, cheaper under the penalty. There's no way, with this design (without standardizing), to distinguish if `history` survived because it's genuinely the most important interaction or simply because it has the largest raw scale — and the observed pattern (only the farthest more dispersed variable survives, all others, including the treatment's main effect, zeroed) strongly weighs in favor of the second explanation.
>
>It's concluded that the result is reported as is, without trying to "fix" via standardization now (would change preprocessing and result at the same time, out of scope for this round — registered as a future methodological extension, not implemented here, Rule #6). The honest reading: a simple linear/additive structure with interactions **can** recover part of the signal of heterogeneity that the shallow tree captures, but the experiment as designed (without standardization) does not isolate which co-variable really carries this interaction — the found candidate (`history`) is more consistent with scale bias than with a substantive discovery. This doesn't change the leader candidate of Section 4 (4.9) nor requires revisiting any previous conclusion — it's reported as an additional exploratory result, purely explanatory.

<a id="s4-summary"></a>

### Synthesis of Section 4

**Fixed-holdout results (one single train→validation split, 4.1-4.8).**

| Item | Result |
|---|---|
| Response-targeting baseline | Qini AUC 0.0395 |
| Best among the 4 original meta-learners (LightGBM vanilla, 4.1-4.3) | S-learner, 0.0415 — slightly ahead of the baseline; X-learner (0.0243) below the baseline, so the structural-advantage hypothesis was not confirmed |
| Heterogeneity diagnostic (4.4) | Real CATE dispersion above the permutation noise floor for the S-learner, and Δ_GATES (top−base) with 95% CI excluding zero for S and X — both exploratory diagnostics, **neither is a formal heterogeneity test** against a non-zero homogeneous ATE (see caveat below) |
| Derived features + regularization in X/R (4.5) | Regularization helps X (0.0243→0.0347); derived features help R (0.0196→0.0352); no configuration beats the original S-learner; combining both axes worsens both |
| Sensitivity to the base algorithm (4.6) | A single shallow tree (`max_depth=4`) beats LightGBM in all 4 meta-learners; X+Tree reaches Qini AUC 0.0627 (+58.7% over the baseline, 1.59×), the highest value in the fixed holdout; Random Forest (vanilla hyperparameters, unrestricted depth) performs poorly (negative Qini for S/T) |
| Known propensity (2/3) vs. estimated in X/R (4.7) | Mostly negative/null result — little change for the X-learner, some help for the R-learner; does not explain the X/R versus S/tree gap |
| Isolating bagging: single tree vs. RF, same depth (4.8) | The RF failure in 4.6 was unrestricted depth, not bagging. With `max_depth=4`, RF becomes positive for all 4 learners; however, bagging does not help consistently and specifically worsens X and T |
| Linear S-learner with T×X interactions (4.10, optional) | Moves away from exact zero (0.0229), but the only interaction coefficient surviving regularization is `T×history` — the largest raw-scale feature among the 18, consistent with an artifact from lack of ElasticNet standardization rather than an identified interaction |

**Repeated stratified holdout results (15 75/25 resamples inside `train_df`, never in `val_df` or the sealed test — 4.9).**

| Item | Result |
|---|---|
| `max_depth` stability (4.9.1) | Mean Qini AUC between 0.0250 and 0.0269 for depth∈{3,4,5}; differences are much smaller than each candidate's standard deviation. No depth shows stable superiority; `max_depth=4` is kept for continuity with 4.6-4.8, not because superiority was demonstrated here |
| Paired comparison of the leader (4.9.2) | X+Tree(depth=4) has the highest mean (0.0250) and win rate (46.7%) among 5 candidates, and beats T+Tree (93% of splits) and both regularized LightGBM variants (67% each) consistently. Against S+LightGBM(vanilla), however, it wins only 53% of splits, with mean Δ of 0.0018 (secondary diagnostics: Wilcoxon p=0.89, paired t-test p=0.71) |
| Leaders vs. response-targeting baseline (4.9.3) | X+Tree beats the baseline in 53% of splits (mean Δ 0.0041); S+LightGBM beats it in 60% (mean Δ 0.0024). Neither shows consistent advantage over the simple response-targeting baseline |
| Qini variability (4.9.4) | Standard deviation is typically 0.010-0.017, large relative to means around 0.02-0.03; plausibly attributed, without formal isolation, to the smaller repeated-protocol sample, relatively low/moderate `visit` prevalence combined with a small incremental effect, and refitting variance at each repetition |

**Central question answered in 4.9.3: do the leading uplift models show a consistent advantage over the simple response-targeting baseline when evaluated on the same resamples? No.** The mean advantage exists and is positive for the two best candidates, but it is small relative to between-sample variability, and the baseline wins nearly half the time. This is reported as observed, without trying to recover a larger advantage through tuning (Absolute Rule #6).

**Conclusions that survived the robustness analysis.**

The T-learner showed strong instability across protocols, not uniformly poor behavior. T+Tree was one of the best fixed-holdout results, with Qini AUC 0.0615 (nearly tied with the leader, 0.0627 — see 4.6/4.8), but fell to mean 0.0039 in repeated stratified holdout and clearly lagged the main competitors (4.9.2, beating the leader in 0% of splits). The excellent performance observed in that fixed split was therefore not robust. This contrast between protocols is itself more interesting and more faithful to the data than simply labeling the T-learner as the "worst model" (the original T-learner with LightGBM in 4.3 was indeed the worst of the four — 0.0067 — but that is a different configuration from T+Tree).

Complexity control of the base learner mattered across configurations. Regularization helped the X- and R-learners (4.5), and limiting the unrestricted Random Forest to `max_depth=4` eliminated the catastrophic behavior seen in the default RF (4.6→4.8). However, those comparisons share the same data, features, and several modeling decisions, so they should not be read as statistically independent evidence that accumulates. The `max_depth ∈ {3,4,5}` stability experiment (4.9.1) also did not identify one specific depth as superior: all three were effectively tied within resampling variability. The correct reading is that controlling base-learner complexity seems important, while the data do not identify one shallow depth as superior.

There is some structure beyond pure sampling noise in the data (permutation noise floor and Δ_GATES, 4.4), but this is not, and should not be read as, formal confirmation that τ(x) varies with X against a non-zero homogeneous ATE.

X-learner + shallow tree and S-learner + LightGBM vanilla form a top tier without stable separation from each other — both remain valid candidates for S5/S6, and neither is eliminated.

**Previous conclusions downgraded or rejected.**

"X-learner + shallow tree (0.0627) is the robust winner of Section 4" — downgraded. The Qini 0.0627 observed in the original holdout was not reproduced under repeated holdout (resampling means stay near 0.025) and showed strong sample/protocol sensitivity. Because the repeated protocol uses fewer observations for both training and evaluation, the full difference cannot be attributed exclusively to a favorable split. But there is also no evidence that the X-learner loses its relative position: it remains in the top tier, just without stable separation from the S-learner.

"The leading meta-learners consistently beat the response-targeting baseline" — rejected with the current data (4.9.3): neither of the two best candidates shows a consistent paired advantage over the baseline on the same resamples.

"Bagging worsens uplift modeling in this dataset" (initial reading from 4.6) — corrected in 4.8: the catastrophic RF result in 4.6 was an artifact of unrestricted depth, not bagging itself. With capped depth, RF is not catastrophic, although it also does not help consistently. It is worth noting that neither the original 4.6 comparison nor its 4.8 correction went through the repeated-resampling robustness analysis used in 4.9; both are fixed-holdout results, not robust dataset properties.

"Real heterogeneity confirmed" (initial reading from 4.4) — corrected: the permutation noise floor tests H0: τ(x)=0 for every x (total null, destroying ATE and heterogeneity together), not H0: τ(x)=ATE (non-zero homogeneous effect). Exceeding this noise floor does not distinguish genuine heterogeneity from a non-zero ATE with asymmetric estimation noise.

"The base algorithm determines the result more than the meta-learning strategy" — softened: base-algorithm choice and regularization had a material impact in this validation, but the current data do not support a stronger deterministic claim.

**Remaining limitations.**

The repeated stratified holdout (not k-fold/OOF) has high Qini AUC variance (4.9.4), with plausible but not formally isolated causes: sample size, outcome prevalence, and model-refitting variance.

The four learners in the Δ_GATES diagnostic (4.4) are not independent evidence — they use the same ~12,800 individuals from `val_df`, with mutually correlated CATE rankings. Two significant results out of four should not be read as "50% independent confirmation."

The linear S-learner with interactions (4.10) is confounded by the lack of ElasticNet feature standardization — unresolved in this round, because it would change preprocessing and result at the same time.

No formal heterogeneity test that preserves the ATE while only destroying dependence on X was implemented. This methodological extension is recorded as future work, not as part of this round.

Everything here is validation/resampling on `train_df`/`val_df`. No conclusion is a formal statistical confirmation; that is the role of S6, with confirmatory evaluation on the sealed test opened exactly once after the final configuration is selected and frozen at the end of S4+S5.

**Direction for S5/S6.** S4 does not select a single winner. X-learner + shallow tree (`max_depth=4`) and S-learner + LightGBM vanilla remain strong references for S5, side by side, without eliminating either for lack of stable separation under resampling. The direct estimators in S5 (Causal Forest, Uplift Trees) will be evaluated under the same development protocol (fixed holdout and repeated stratified holdout) and compared with these references and with the response-targeting baseline; they do not compete against a predefined S4 "winner." The final configuration will be selected and frozen at the end of S5 using only development data. If candidates remain practically tied at the end of S5, the tie-breaking criterion or confirmatory comparison set will be defined explicitly before the sealed test is opened. In S6, the sealed test will be opened exactly once, only for confirmatory evaluation of the pre-selected configuration and explicitly pre-specified comparisons; the test does not participate in model selection.

**S4 is frozen from here onward:** no new depth test, Random Forest variant, feature, propensity variant, or additional meta-learner will be added to this section.

**Next:** Section 5 — Causal Forest and Uplift Trees.


---

<a id="glossary"></a>

## Glossary (Quick Reference)

| Term | Definition |
|---|---|
| **ATE** | Average Treatment Effect — average treatment effect on the population |
| **ITE** | Individual Treatment Effect — individual effect (unobservable) |
| **CATE** | Conditional Average Treatment Effect — average treatment effect conditional on $X$, the target of uplift modeling |
| **SMD** | Standardized Mean Difference — metric of balance between groups |
| **Uplift score** | Estimate of $\tau(x)$ produced by a model |
| **Persuadable** | Customer with significant positive uplift |
| **Sure Thing** | Customer who converts with or without treatment (uplift ≈ 0) |
| **Lost Cause** | Customer who does not convert in any scenario (uplift ≈ 0) |
| **Sleeping Dog** | Customer with negative uplift — treatment is detrimental |
| **Qini coefficient** | Metric of uplift evaluation analogous to Gini, integrates the Qini curve |
| **AUUC** | Area Under the Uplift Curve — equivalent to AUC for uplift |